In [1]:
import torch

# Limit to fraction of total GPU memory
torch.cuda.set_per_process_memory_fraction(0.5, device=0)

# Training

Load the data

In [2]:
# Path to train data
train_data_dir = '/home/elek/sds/sd17d003/Anamaria/splicevo/data/splits_small/mouse_human/train/'

import os
import pandas as pd

# First load metadat
meta_csv_fn = os.path.join(train_data_dir, 'metadata.csv')
metadata_csv = pd.read_csv(meta_csv_fn)
print(metadata_csv.head())

import json

meta_fn = os.path.join(train_data_dir, 'metadata.json')
with open(meta_fn) as f:
    metadata = json.load(f)

print("\nWindow size", metadata.get('window_size', 0))
print("Context size", metadata.get('context_size', 0))
species_conds = metadata.get('species_condition_mapping')

for org, cond in species_conds.items():
    print(f"Conditions for {org}:")
    print([metadata['usage_conditions'][cond_idx] for cond_idx in cond])

      genome_id  chromosome    gene_id strand  gene_start  gene_end  \
0  human_GRCh37          11  hum.10002      +    18433956  18472509   
1  human_GRCh37          11  hum.10012      +    18271664  18291263   
2  human_GRCh37          11  hum.10013      -    18288526  18291263   
3  human_GRCh37          11  hum.10006      -    18134190  18285796   
4  human_GRCh37          11  hum.10017      +    18344287  18388847   

   gene_length  sequence_start  sequence_end  sequence_length  context_size  \
0        38554        18432956      18473509            40554          1000   
1        19600        18270664      18292263            21600          1000   
2         2738        18287526      18292263             4738          1000   
3       151607        18133190      18286796           153607          1000   
4        44561        18343287      18389847            46561          1000   

   n_donor_sites  n_acceptor_sites  species_id  
0              8                 7           0  


Load the model

In [3]:
import yaml
import torch
import torch.nn as nn
from alphagenome_pytorch import AlphaGenome, AlphaGenomeConfig
from pathlib import Path

# Initialize the model
default_cfg = AlphaGenomeConfig()

config_file = f"../configs/splice_finetune_small.yaml"
with open(config_file, 'r') as f:
    config = yaml.safe_load(f)
model_cfg = config.get('model', {})
dims = tuple(model_cfg.get('dims', default_cfg.dims))
basepairs = model_cfg.get('basepairs', default_cfg.basepairs)
dna_embed_width = model_cfg.get('dna_embed_width', default_cfg.dna_embed_width)
num_organisms = model_cfg.get('num_organisms', default_cfg.num_organisms)
transformer_kwargs = model_cfg.get('transformer_kwargs', default_cfg.transformer_kwargs)

model_pretrained = AlphaGenome(dims, basepairs, dna_embed_width, num_organisms, transformer_kwargs)

# Add heads
model_pretrained.add_reference_heads("human")
model_pretrained.add_reference_heads('mouse')

# Heads configurations
heads_cfg = model_cfg.get('heads_cfg', {
    'human': {
        'num_tracks_1bp': 0,
        'num_tracks_128bp': 0,
        'num_tracks_contacts': 0,
        'num_splicing_contexts': 16
    },
    'mouse': {
        'num_tracks_1bp': 0,
        'num_tracks_128bp': 0,
        'num_tracks_contacts': 0,
        'num_splicing_contexts': 34
    }
})
# Load pretrained weights
pretrained_model_version = "all_folds"
cache_dir = "../outputs/checkpoints/pretrained"
print(f"Loading pretrained weights from {pretrained_model_version}...")
cache_dir = Path(cache_dir)
cache_dir.mkdir(exist_ok=True, parents=True)
cached_model_path = cache_dir / f'pretrained_model_{pretrained_model_version}.pt'
if cached_model_path.exists():
    print(f"Loading pretrained model from cache: {cached_model_path}")
    checkpoint = torch.load(cached_model_path, map_location='cpu')
    model_pretrained.load_state_dict(checkpoint['model_state_dict'])
else:
    print("No cached model found. Downloading pretrained weights...")
    model_pretrained.load_from_official_jax_model(pretrained_model_version, strict=False)
    print("Saving to cache for future use...")
    torch.save({'model_state_dict': model_pretrained.state_dict()}, cached_model_path)
    print(f"Model cached to {cached_model_path}")

# Weights and biases
weights_and_biases = dict()
for organism in ['human', 'mouse']:
    weights_and_biases[organism] = dict()
    w = model_pretrained.heads[organism]['splice_sites_classification'].linear.weight.data.clone()
    b = model_pretrained.heads[organism]['splice_sites_classification'].linear.bias.data.clone()
    weights_and_biases[organism]['classification_head'] = dict()
    weights_and_biases[organism]['classification_head']['weight'] = w
    weights_and_biases[organism]['classification_head']['bias'] = b
    print(f"Output head weights: ")
    print(f"  {organism} splice classification head weight mean: {w.mean().item():.6f}, std: {w.std().item():.6f}")
    print(f"  {organism} splice classification head bias mean: {b.mean().item():.6f}, std: {b.std().item():.6f}")
print("Pretrained weights loaded successfully")

# Remove existing heads and add new splicing heads
model_pretrained.heads = torch.nn.ModuleDict()
for organism, head_cfg in heads_cfg.items():
    model_pretrained.add_heads(organism=organism, **head_cfg)

# Copy output head weights from pretrained model
print("Copying pretrained weights after adding new heads...")
for org_name in ['human', 'mouse']:
    model_pretrained.heads[org_name]['splice_sites_classification'].linear.weight.data.copy_(weights_and_biases[org_name]['classification_head']['weight'])
    model_pretrained.heads[org_name]['splice_sites_classification'].linear.bias.data.copy_(weights_and_biases[org_name]['classification_head']['bias'])


2026-03-09 11:57:59.254377: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 11:57:59.407558: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading pretrained weights from all_folds...
Loading pretrained model from cache: ../outputs/checkpoints/pretrained/pretrained_model_all_folds.pt
Output head weights: 
  human splice classification head weight mean: 0.000454, std: 0.137590
  human splice classification head bias mean: 0.758246, std: 0.952418
Output head weights: 
  mouse splice classification head weight mean: 0.003590, std: 0.144426
  mouse splice classification head bias mean: 0.194314, std: 0.286608
Pretrained weights loaded successfully
Copying pretrained weights after adding new heads...


/home/elek/miniforge3/envs/alphagenome_pytorch/lib/python3.12/site-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


In [4]:
%reload_ext autoreload
%autoreload 2

# Load training data
import numpy as np
from torch.utils.data import DataLoader, Subset
from alphagenome_pytorch.data.splice_dataset import SpliceDataset

# Load dataset
train_dataset = SpliceDataset(
    data_dir=train_data_dir,
    target_length=None,
    max_donor_sites=20,
    max_acceptor_sites=20,
    species_mapping={'human': 0, 'mouse': 1}
)

Loaded dataset from /home/elek/sds/sd17d003/Anamaria/splicevo/data/splits_small/mouse_human/train/


In [5]:
next(iter(train_dataset))

{'dna': tensor([ 2,  0,  2,  ..., -1, -1, -1]),
 'organism_index': tensor(0),
 'splice_donor_idx': tensor([ 1998,  2052,  2433,  4891, 19500, 24503, 28235, 28235, 28235, 28235,
         28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235]),
 'splice_acceptor_idx': tensor([ 2299,  4774, 19327, 24330, 28118, 28118, 28118, 28118, 28118, 28118,
         28118, 28118, 28118, 28118, 28118, 28118, 28118, 28118, 28118, 28118]),
 'num_donors': tensor(7),
 'num_acceptors': tensor(5),
 'splice_labels': tensor([4, 4, 4,  ..., 4, 4, 4]),
 'splice_usage_target': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 'conditions_mask': tensor([0, 1, 2, 3, 5, 6, 7]),
 'gene_region': tensor([    0, 40960])}

In [6]:
from alphagenome_pytorch.samplers import SpeciesAndLengthGroupedSampler

# Use small subset for demonstration (first n samples)
n = 20
train_subset = Subset(train_dataset, np.arange(min(n, len(train_dataset))))

batch_size = 4
sampler = SpeciesAndLengthGroupedSampler(train_dataset, batch_size=batch_size, shuffle=False)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=sampler,
    num_workers=0,
    pin_memory=False
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Batches per epoch: {len(train_loader)}")

Dataset size: 5052
Batch size: 4
Batches per epoch: 1482


In [7]:
next(iter(train_loader))

{'dna': tensor([[ 2,  0,  2,  ..., -1, -1, -1],
         [ 1,  1,  1,  ..., -1, -1, -1],
         [ 2,  3,  1,  ..., -1, -1, -1],
         [ 3,  3,  1,  ..., -1, -1, -1]]),
 'organism_index': tensor([0, 0, 0, 0]),
 'splice_donor_idx': tensor([[ 1998,  2052,  2433,  4891, 19500, 24503, 28235, 28235, 28235, 28235,
          28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235, 28235],
         [ 1998,  5159,  6744, 22906, 24363, 26204, 29068, 33567, 33567, 33567,
          33567, 33567, 33567, 33567, 33567, 33567, 33567, 33567, 33567, 33567],
         [ 8483, 10652, 10960, 11013, 11109, 11300, 13406, 20570, 25579, 25579,
          25579, 25579, 25579, 25579, 25579, 25579, 25579, 25579, 25579, 25579],
         [ 1998, 23497, 27454, 29851, 32380, 32575, 32575, 32575, 32575, 32575,
          32575, 32575, 32575, 32575, 32575, 32575, 32575, 32575, 32575, 32575]]),
 'splice_acceptor_idx': tensor([[ 2299,  4774, 19327, 24330, 28118, 28118, 28118, 28118, 28118, 28118,
          28118, 

In [8]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

# Set up optimizer and loss
model_pretrained = model_pretrained.to(device)

# Create loss function
loss_fn = nn.CrossEntropyLoss()

# Create optimizer
lr = 1e-5
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_pretrained.parameters()),
    lr=lr,
    weight_decay=0.01
)

print(f"Optimizer: AdamW with LR={lr}")
print(f"Loss function: CrossEntropyLoss")
print(f"Trainable parameters: {sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad):,}")

# Store reference to initial weights for comparison
initial_weights = {}
for org_name in ['human', 'mouse']:
    w = model_pretrained.heads[org_name]['splice_sites_classification'].linear.weight.data.clone()
    b = model_pretrained.heads[org_name]['splice_sites_classification'].linear.bias.data.clone()
    initial_weights[org_name] = {'weight': w, 'bias': b}
    print(f"\nInitial {org_name} head weights: mean={w.mean().item():.6f}, std={w.std().item():.6f}")
    print(f"Initial {org_name} head bias: mean={b.mean().item():.6f}, std={b.std().item():.6f}")

# Update running variance
update_running_var = False


Device: cuda
Optimizer: AdamW with LR=1e-05
Loss function: CrossEntropyLoss
Trainable parameters: 405,516,192

Initial human head weights: mean=0.000454, std=0.137590
Initial human head bias: mean=0.758246, std=0.952418

Initial mouse head weights: mean=0.003590, std=0.144426
Initial mouse head bias: mean=0.194314, std=0.286608


In [10]:
# Batch-by-batch training simulation with weight and classification metrics tracking
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score

class_names = ['dononr_plus', 'acceptor_plus', 'donor_minus', 'acceptor_minus', 'non_splice_site']

# Initialize tracking dictionaries
training_history = {
    'batch': [],
    'loss': [],
    'human_weight_mean': [],
    'human_weight_std': [],
    'human_bias_mean': [],
    'mouse_weight_mean': [],
    'mouse_weight_std': [],
    'mouse_bias_mean': [],
    'class_dist': [],
    'true_class_dist_all': [],
    'true_class_dist_masked': [],
    'precision_per_class': [],
    'recall_per_class': [],
    'auroc_per_class': []
}

# Training loop
num_batches_to_show = min(10, len(train_loader))  # Train for 10 batches max for demo
model_pretrained.train()

print(f"Training for {num_batches_to_show} batches with weighted loss...\n")

from alphagenome_pytorch.alphagenome import set_update_running_var
set_update_running_var(model_pretrained, update_running_var)

species_mapping = {'human': 0, 'mouse': 1}

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= num_batches_to_show:
        break

    # Get batch data
    dna = batch['dna'].to(device)
    organism_index = batch['organism_index'].to(device)
    splice_donor_idx = batch['splice_donor_idx'].to(device)
    splice_acceptor_idx = batch['splice_acceptor_idx'].to(device)
    splice_labels = batch['splice_labels'].to(device)
    gene_region = batch['gene_region'].to(device)

    # Forward pass
    preds = model_pretrained(
        dna,
        organism_index,
        splice_donor_idx=splice_donor_idx,
        splice_acceptor_idx=splice_acceptor_idx
    )

    # Compute loss (same masking as training)
    losses = []

    # Collect masked labels/probabilities for per-batch metrics
    batch_masked_true_labels = []
    batch_masked_probs = []

    for org_name, org_preds in preds.items():
        if 'splice_sites_classification' not in org_preds:
            continue

        splice_logits = org_preds['splice_sites_classification']
        _, seq_len_actual, _ = splice_logits.shape

        # Filter by organism
        org_idx = species_mapping[org_name]
        org_mask = organism_index == org_idx
        if not org_mask.any():
            continue

        org_gene_region = gene_region[org_mask]
        org_splice_labels = splice_labels[org_mask]

        # Create mask for positions in gene regions
        upper_bound = min(seq_len_actual, 23000)
        starts = org_gene_region[:, 0].clamp(0, upper_bound)
        ends = org_gene_region[:, 1].clamp(0, upper_bound)
        positions = torch.arange(seq_len_actual, device=device).unsqueeze(0)
        mask = (positions >= starts.unsqueeze(1)) & (positions < ends.unsqueeze(1))

        if mask.any():
            masked_true = org_splice_labels[mask]
            masked_logits = splice_logits[mask]
            masked_probs = torch.softmax(masked_logits, dim=-1)

            batch_masked_true_labels.append(masked_true)
            batch_masked_probs.append(masked_probs)

            loss = loss_fn(masked_logits, masked_true)
            losses.append(loss)

    if len(losses) > 0:
        loss = torch.stack(losses).sum()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pretrained.parameters(), max_norm=1.0)
        optimizer.step()

        # Track metrics
        training_history['batch'].append(batch_idx + 1)
        training_history['loss'].append(loss.item())

        # Track weight changes
        for org_name in ['human', 'mouse']:
            w = model_pretrained.heads[org_name]['splice_sites_classification'].linear.weight.data
            b = model_pretrained.heads[org_name]['splice_sites_classification'].linear.bias.data

            if org_name == 'human':
                training_history['human_weight_mean'].append(w.mean().item())
                training_history['human_weight_std'].append(w.std().item())
                training_history['human_bias_mean'].append(b.mean().item())
            else:
                training_history['mouse_weight_mean'].append(w.mean().item())
                training_history['mouse_weight_std'].append(w.std().item())
                training_history['mouse_bias_mean'].append(b.mean().item())

        # Per-batch class counts and metrics
        with torch.no_grad():
            # True counts across ALL positions
            class_dist_true_all = [(splice_labels == c).sum().item() for c in range(5)]

            # True counts + predictions in MASKED positions only
            if len(batch_masked_true_labels) > 0:
                masked_true_concat = torch.cat(batch_masked_true_labels)
                masked_probs_concat = torch.cat(batch_masked_probs)
                masked_pred_concat = torch.argmax(masked_probs_concat, dim=-1)

                class_dist_true_masked = [(masked_true_concat == c).sum().item() for c in range(5)]
                class_dist_pred = [(masked_pred_concat == c).sum().item() for c in range(5)]

                # Precision / Recall / AUROC per class (one-vs-rest)
                precision_per_class = []
                recall_per_class = []
                auroc_per_class = []

                y_true_np = masked_true_concat.cpu().numpy()
                y_prob_np = masked_probs_concat.cpu().numpy()
                y_pred_np = masked_pred_concat.cpu().numpy()

                for c in range(5):
                    tp = np.sum((y_pred_np == c) & (y_true_np == c))
                    fp = np.sum((y_pred_np == c) & (y_true_np != c))
                    fn = np.sum((y_pred_np != c) & (y_true_np == c))

                    precision_c = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                    recall_c = tp / (tp + fn) if (tp + fn) > 0 else 0.0

                    y_bin = (y_true_np == c).astype(np.int32)
                    if len(np.unique(y_bin)) < 2:
                        auroc_c = np.nan
                    else:
                        auroc_c = float(roc_auc_score(y_bin, y_prob_np[:, c]))

                    precision_per_class.append(float(precision_c))
                    recall_per_class.append(float(recall_c))
                    auroc_per_class.append(float(auroc_c) if np.isfinite(auroc_c) else np.nan)

            else:
                class_dist_true_masked = [0, 0, 0, 0, 0]
                class_dist_pred = [0, 0, 0, 0, 0]
                precision_per_class = [0.0] * 5
                recall_per_class = [0.0] * 5
                auroc_per_class = [np.nan] * 5

            training_history['class_dist'].append(class_dist_pred)
            training_history['true_class_dist_all'].append(class_dist_true_all)
            training_history['true_class_dist_masked'].append(class_dist_true_masked)
            training_history['precision_per_class'].append(precision_per_class)
            training_history['recall_per_class'].append(recall_per_class)
            training_history['auroc_per_class'].append(auroc_per_class)

        # Print progress
        print(f"Batch {batch_idx + 1}/{num_batches_to_show} - Loss: {loss.item():.6f}")
        print(f"  Human head: weight_mean={training_history['human_weight_mean'][-1]:.6f}, "
              f"weight_std={training_history['human_weight_std'][-1]:.6f}")
        print(f"  Mouse head: weight_mean={training_history['mouse_weight_mean'][-1]:.6f}, "
              f"weight_std={training_history['mouse_weight_std'][-1]:.6f}")

        print("  True (all positions):", end=" ")
        for c, count in enumerate(class_dist_true_all):
            print(f"{class_names[c]} {count}", end="; ")
        print()

        print("  True (masked only):", end=" ")
        for c, count in enumerate(class_dist_true_masked):
            print(f"{class_names[c]} {count}", end="; ")
        print()

        print("  Predictions (masked):", end=" ")
        for c, count in enumerate(class_dist_pred):
            print(f"{class_names[c]} {count}", end="; ")
        print()

        print("  Per-class metrics (masked):")
        for c in range(5):
            auc_str = "N/A" if not np.isfinite(auroc_per_class[c]) else f"{auroc_per_class[c]:.3f}"
            print(f"    {class_names[c]:15s} | P={precision_per_class[c]:.3f} R={recall_per_class[c]:.3f} AUROC={auc_str}")
        print()

print("Training simulation complete!")


Training for 10 batches with weighted loss...

Batch 1/10 - Loss: 0.005845
  Human head: weight_mean=0.000454, weight_std=0.137586
  Mouse head: weight_mean=0.003590, weight_std=0.144426
  True (all positions): dononr_plus 21; acceptor_plus 17; donor_minus 9; acceptor_minus 10; non_splice_site 163783; 
  True (masked only): dononr_plus 10; acceptor_plus 6; donor_minus 8; acceptor_minus 9; non_splice_site 91967; 
  Predictions (masked): dononr_plus 7; acceptor_plus 6; donor_minus 2; acceptor_minus 2; non_splice_site 91983; 
  Per-class metrics (masked):
    dononr_plus     | P=0.000 R=0.000 AUROC=0.375
    acceptor_plus   | P=0.000 R=0.000 AUROC=0.469
    donor_minus     | P=0.000 R=0.000 AUROC=0.814
    acceptor_minus  | P=0.000 R=0.000 AUROC=0.700
    non_splice_site | P=1.000 R=1.000 AUROC=0.510

Batch 2/10 - Loss: 0.013143
  Human head: weight_mean=0.000454, weight_std=0.137584
  Mouse head: weight_mean=0.003590, weight_std=0.144426
  True (all positions): dononr_plus 21; acceptor_p

In [11]:
# Print summary
print("Training summary")
print(f"\nHuman Head Classification Weights:")
print(f"  Initial mean: {initial_weights['human']['weight'].mean().item():.6f}")
print(f"  Final mean:   {training_history['human_weight_mean'][-1]:.6f}")
print(f"  Change:       {training_history['human_weight_mean'][-1] - initial_weights['human']['weight'].mean().item():.6f}")
print(f"  Initial std:  {initial_weights['human']['weight'].std().item():.6f}")
print(f"  Final std:    {training_history['human_weight_std'][-1]:.6f}")

print(f"\nMouse Head Classification Weights:")
print(f"  Initial mean: {initial_weights['mouse']['weight'].mean().item():.6f}")
print(f"  Final mean:   {training_history['mouse_weight_mean'][-1]:.6f}")
print(f"  Change:       {training_history['mouse_weight_mean'][-1] - initial_weights['mouse']['weight'].mean().item():.6f}")
print(f"  Initial std:  {initial_weights['mouse']['weight'].std().item():.6f}")
print(f"  Final std:    {training_history['mouse_weight_std'][-1]:.6f}")

print(f"\nLoss:")
print(f"  Batch 1: {training_history['loss'][0]:.6f}")
print(f"  Batch {len(training_history['loss'])}: {training_history['loss'][-1]:.6f}")
print(f"  Reduction: {(1 - training_history['loss'][-1] / training_history['loss'][0]) * 100:.2f}%")

print(f"\nPer-class metrics (masked) - Batch 1 vs Batch {len(training_history['loss'])}:")
precision_first = training_history['precision_per_class'][0]
precision_last = training_history['precision_per_class'][-1]
recall_first = training_history['recall_per_class'][0]
recall_last = training_history['recall_per_class'][-1]
auroc_first = training_history['auroc_per_class'][0]
auroc_last = training_history['auroc_per_class'][-1]

for cls in range(5):
    auc_first_str = "N/A" if not np.isfinite(auroc_first[cls]) else f"{auroc_first[cls]:.3f}"
    auc_last_str = "N/A" if not np.isfinite(auroc_last[cls]) else f"{auroc_last[cls]:.3f}"
    print(
        f"  {class_names[cls]:15s} | "
        f"P: {precision_first[cls]:.3f} -> {precision_last[cls]:.3f} | "
        f"R: {recall_first[cls]:.3f} -> {recall_last[cls]:.3f} | "
        f"AUROC: {auc_first_str} -> {auc_last_str}"
    )

print("\nPer-batch macro averages (masked):")
for i, batch_id in enumerate(training_history['batch']):
    p_vals = np.array(training_history['precision_per_class'][i], dtype=float)
    r_vals = np.array(training_history['recall_per_class'][i], dtype=float)
    a_vals = np.array(training_history['auroc_per_class'][i], dtype=float)

    p_macro = float(np.nanmean(p_vals))
    r_macro = float(np.nanmean(r_vals))
    a_macro = float(np.nanmean(a_vals)) if np.any(np.isfinite(a_vals)) else np.nan

    a_macro_str = "N/A" if not np.isfinite(a_macro) else f"{a_macro:.3f}"
    print(f"  Batch {batch_id:2d} | Precision={p_macro:.3f} Recall={r_macro:.3f} AUROC={a_macro_str}")


Training summary

Human Head Classification Weights:
  Initial mean: 0.000454
  Final mean:   0.000453
  Change:       -0.000001
  Initial std:  0.137590
  Final std:    0.137569

Mouse Head Classification Weights:
  Initial mean: 0.003590
  Final mean:   0.003590
  Change:       0.000000
  Initial std:  0.144426
  Final std:    0.144426

Loss:
  Batch 1: 0.005845
  Batch 10: 0.007178
  Reduction: -22.80%

Per-class metrics (masked) - Batch 1 vs Batch 10:
  dononr_plus     | P: 0.000 -> 0.000 | R: 0.000 -> 0.000 | AUROC: 0.375 -> 0.553
  acceptor_plus   | P: 0.000 -> 0.000 | R: 0.000 -> 0.000 | AUROC: 0.469 -> 0.419
  donor_minus     | P: 0.000 -> 0.000 | R: 0.000 -> 0.000 | AUROC: 0.814 -> 0.751
  acceptor_minus  | P: 0.000 -> 0.000 | R: 0.000 -> 0.000 | AUROC: 0.700 -> 0.781
  non_splice_site | P: 1.000 -> 0.999 | R: 1.000 -> 1.000 | AUROC: 0.510 -> 0.583

Per-batch macro averages (masked):
  Batch  1 | Precision=0.200 Recall=0.200 AUROC=0.574
  Batch  2 | Precision=0.200 Recall=0.20

## Log off

In [12]:
# Free uo any used memory
with torch.no_grad():
    torch.cuda.empty_cache()